In [1]:
with open("./input.txt", "r") as f:
    text = f.read()

In [2]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:
vocab = sorted(list(set(text)))
vocab_size = len(vocab)
print(''.join(vocab))
vocab_size


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


65

In [4]:
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [5]:
sentence = "hi there frend"
print(encode(sentence))
print(decode(encode(sentence)))

[46, 47, 1, 58, 46, 43, 56, 43, 1, 44, 56, 43, 52, 42]
hi there frend


use torch

In [6]:
import torch
torch.manual_seed(1337)

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:50])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [8]:
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

we get the context window as input in increasing order \
if [18, ] is context, target is 47; \
if [18, 47, ] is context, target is 56 and so on.. \
this is done for all blocks generated to make the model used to seeing all varying sized contexts \
after block_size, we need to truncate

In [10]:
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [11]:
xb, yb = get_batch("train")
xb, yb

(tensor([[58, 63,  8,  0,  0, 19, 24, 27],
         [39, 59, 45, 46, 58,  1, 46, 43],
         [49, 43, 57,  1, 53, 50, 42,  1],
         [52, 41, 47, 43, 52, 58,  1, 56]]),
 tensor([[63,  8,  0,  0, 19, 24, 27, 33],
         [59, 45, 46, 58,  1, 46, 43,  1],
         [43, 57,  1, 53, 50, 42,  1, 46],
         [41, 47, 43, 52, 58,  1, 56, 47]]))

In [12]:
for b in range(batch_size):
    for t in range(block_size):
        ctx = xb[b, :t+1]
        target = yb[b, t]
        print(f"input is {ctx.tolist()} target: {target}")

input is [58] target: 63
input is [58, 63] target: 8
input is [58, 63, 8] target: 0
input is [58, 63, 8, 0] target: 0
input is [58, 63, 8, 0, 0] target: 19
input is [58, 63, 8, 0, 0, 19] target: 24
input is [58, 63, 8, 0, 0, 19, 24] target: 27
input is [58, 63, 8, 0, 0, 19, 24, 27] target: 33
input is [39] target: 59
input is [39, 59] target: 45
input is [39, 59, 45] target: 46
input is [39, 59, 45, 46] target: 58
input is [39, 59, 45, 46, 58] target: 1
input is [39, 59, 45, 46, 58, 1] target: 46
input is [39, 59, 45, 46, 58, 1, 46] target: 43
input is [39, 59, 45, 46, 58, 1, 46, 43] target: 1
input is [49] target: 43
input is [49, 43] target: 57
input is [49, 43, 57] target: 1
input is [49, 43, 57, 1] target: 53
input is [49, 43, 57, 1, 53] target: 50
input is [49, 43, 57, 1, 53, 50] target: 42
input is [49, 43, 57, 1, 53, 50, 42] target: 1
input is [49, 43, 57, 1, 53, 50, 42, 1] target: 46
input is [52] target: 41
input is [52, 41] target: 47
input is [52, 41, 47] target: 43
input is

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [26]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [35]:
m = BigramLanguageModel(vocab_size)
out, loss = m(xb, yb)
print(out.shape, loss)

torch.Size([256, 65]) tensor(4.7328, grad_fn=<NllLossBackward0>)


In [36]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, 100)[0].tolist()))


SMcrXXo,'DJCEq3wO..x$zdebwZlSAmzxANFQII
3Egpjbipjanhe;IvFDLTemGcaswfVP.PpjZqI.x3QIjAA hPOAz
nyIFKCac


Training

In [37]:
# opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
opt = torch.optim.AdamW(m.parameters(), lr=2e-3)

In [38]:
batch_size = 32
for steps in range(5000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print(loss.item())

2.5408802032470703


In [41]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, 300)[0].tolist()))


LL:
Frcavthe yo 'G dseyomod s m as, rinthitt gl waseritheiman, mbate atht man hy ron y fou fuber amnede it liof s'S:RICAntrd my ipl th; ad hes nd pe d,
ULOOpade s sstccine wh spey, sst
TULEENGomoone
Than,qusite by?
Cou wod
NGUCiseld bas'sh cemy tlest,


Be
RDUS: clito cen O CIARYourengo herd lds hem


uptil the above Bigram model, tokens are not talking to each other, just one sized context is used!